# SpectraShift Week 6: seed 29
Run 15 missing curve jobs and the M1RGB control sequentially. Attach source v5, Week 2 frozen data, Week 5 contracts, the verified Week 5 pilot summary, Week 5 complete, Week 6 contracts, and `spectrashift-week4-seed29`. Use GPU T4 x2 with Internet off. This is notebook 12b.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week6.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 6 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week6-source')
    if source_work.exists():
        shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 6 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        by_hash.setdefault(digest, path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

SEED = 29
WORK = Path('/kaggle/working/spectrashift-week6-seed29')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
PILOT_SUMMARY = unique_file('week5_pilot_summary.json')
WEEK5_SUMMARY = unique_file('week5_run_summary.json')
WEEK6_CONTRACTS = unique_file('week6_contracts_summary.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
config = yaml.safe_load((PROJECT / 'configs/downstream/week6.yaml').read_text())
config['data']['manifest_path'] = str(MANIFEST)
config['data']['staged_root'] = str(STAGED)
config['data']['normalization_path'] = str(NORMALIZATION)
config['data']['freeze_summary_path'] = str(FREEZE)
config['contracts']['week5_contracts_summary_path'] = str(WEEK5_CONTRACTS)
config['contracts']['week5_summary_path'] = str(WEEK5_SUMMARY)
RUNTIME_CONFIG = WORK / 'week6.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
import torch
encoder_paths = {}
for path in INPUT.rglob('encoder-final.pt'):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if payload.get('model_id') in {'M2','M3','M4'}:
        encoder_paths[str(payload['run_id'])] = path
expected = {f'week4-{model}-seed29' for model in ('m2','m3','m4')}
assert set(encoder_paths) == expected, f'Expected {expected}, found {set(encoder_paths)}'
print({'project': str(PROJECT), 'work': str(WORK), 'encoders': sorted(encoder_paths)})


In [ ]:
assert torch.cuda.is_available(), 'Select GPU T4 x2 before running'
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
assert 'T4' in gpu_name and f'sm_{major}{minor}' in torch.cuda.get_arch_list()
print({'gpu': gpu_name, 'gpu_count': torch.cuda.device_count(), 'using_device': 0})


In [ ]:
from spectrashift.train.week6 import run_week6_seed

summary = run_week6_seed(
    RUNTIME_CONFIG, SEED, WORK, WEEK5_CONTRACTS, PILOT_SUMMARY,
    WEEK5_SUMMARY, WEEK6_CONTRACTS, encoder_paths, resume_roots=[INPUT],
)
print(json.dumps(summary, indent=2))
assert summary['week6_seed_complete'] and summary['run_count'] == 16
assert summary['evaluation_labels_loaded'] is False
